# Mathematical Engineering - Financial Engineering, FY 2025-2026

# Risk Management - Assignment 1: Hedging a Swaption Portfolio

**Case study:** The IR-derivative desk of Polimi Bank has the following positions opened today (15/02/2008):

- Long swaption payer 1m&10y 5y ATM - Notional €650 Mln
- Vanilla 10y IRS fixed rate receiver - Notional €550 Mln


In [4]:
# Importing the libraries
import numpy as np
import pandas as pd
import pickle
import datetime as dt

from date_functions import (
    business_date_offset,
    date_series,
    year_frac_act_x
)
from ex0_utilities import bootstrap
from ex1_utilities import (
    swaption_price_calculator,
    irs_proxy_duration,
    swap_par_rate,
    swap_mtm,
    SwapType,
)

In [ ]:
# Portfolio Parameters

# Swaption
swaption_maturity_y = 10  # Years component of swaption expiry
swaption_maturity_m = 1  # Months component of swaption expiry, modified to 1 
swaption_tenor_y = 5  # Underlying swap tenor in years
swaption_fixed_leg_freq = 1  # Annual fixed leg payments
swaption_type = SwapType.PAYER
swaption_notional = 650_000_000
sigma_black = 0.7955  # Black implied volatility for the swaption

# IRS
irs_maturity = 10  # In years
irs_fixed_leg_freq = 1  # Annual fixed leg payments
irs_notional = 550_000_000
irs_type = SwapType.RECEIVER # it was payer we've modified it 

In [7]:
dt = pd.read_csv('dt.csv',
                index_col = 'Market',
                usecols = ['Market','TARGET'],
                converters = {'TARGET':pd.to_datetime})

depo_converter = lambda x: float(x)
df_depos = pd.read_csv('depos.csv', 
                   index_col ='Depos',
                   usecols = ['Depos','ASK','BID'], 
                    converters={'Depos':pd.to_datetime,'BID':depo_converter,'ASK':depo_converter})

future_converter = lambda x: float(x)
futures = pd.read_csv('futures.csv',
                      index_col ='Futures',
                      usecols = ['Futures','ASK','BID'],
                      converters = {'Futures':pd.to_datetime, 'Settle':pd.to_datetime, 'Expiry':pd.to_datetime})
expiry = pd.read_csv('expiry.csv',
                     index_col = 'Futures',
                     usecols =['Futures', 'Settle', 'Expiry'], 
                     converters = {'Futures':pd.to_datetime, 'Settle':pd.to_datetime, 'Expiry':pd.to_datetime})
df_futures = futures.join(expiry)

swap_converter = lambda x: float(x)
df_swaps = pd.read_csv('swaps.csv',
                    index_col = 'Swaps',
                    usecols = ['Swaps','BID','ASK'],
                    converters={'Swaps':pd.to_datetime,'BID':swap_converter,'ASK':swap_converter})

today = dt.TARGET['Today']
settlement_date  = dt.TARGET['Settlement']

# Storing the data in a dictionary
market_data = dict()
market_data["reference_date"] = settlement_date
market_data["depo"] = df_depos
market_data["futures"] = df_futures
market_data["swaps"] = df_swaps
pickle.dump(market_data, open("market_data.p", "wb"))


# Bootstrap
discount_factors, zero_rates = bootstrap(settlement_date, df_depos, df_futures, df_swaps)

## Q1: Mark-to-Market the portfolio at the mid-rate curve


In [ ]:
# =============================================================================
# Q1: Compute the forward swap rate (= swaption strike, since ATM)
# =============================================================================

# every today was substituted with settlment date 

# Swaption expiry: today + 10y1m
swaption_expiry = business_date_offset(
    settlement_date, year_offset=swaption_maturity_y, month_offset=swaption_maturity_m
)

# Underlying swap expiry: swaption expiry + 5y tenor
underlying_expiry = business_date_offset(
    settlement_date,
    year_offset=swaption_maturity_y + swaption_tenor_y,
    month_offset=swaption_maturity_m,
)

# Fixed leg schedule of the underlying forward-starting swap
swaption_underlying_fixed_leg_schedule = date_series(
    swaption_expiry, underlying_expiry, swaption_fixed_leg_freq
)

# Forward swap rate
fwd_swap_rate = swap_par_rate(
    swaption_underlying_fixed_leg_schedule[1:],
    discount_factors,
    swaption_underlying_fixed_leg_schedule[0],
)
print(f"Forward swap rate: {fwd_swap_rate:.3%}")


Forward swap rate: 5.300%


In [49]:
# =============================================================================
# Q1: Portfolio MtM
# =============================================================================

strike = fwd_swap_rate # becuase ATM swaption

swaption_price, swaption_delta = swaption_price_calculator(
    fwd_swap_rate,
    strike,
    settlement_date, # it was today, changed to settlemnt date
    swaption_expiry,
    underlying_expiry,
    sigma_black,
    swaption_fixed_leg_freq,
    discount_factors,
    swaption_type,
    compute_delta=True,
)

# today changed with settlemnt date 
irs_expiry = business_date_offset(settlement_date, year_offset=irs_maturity)
irs_fixed_leg_payment_dates = date_series(settlement_date, irs_expiry, irs_fixed_leg_freq)[1:]

irs_rate = swap_par_rate(irs_fixed_leg_payment_dates, discount_factors)
irs_mtm = swap_mtm(
    irs_rate, irs_fixed_leg_payment_dates, discount_factors
)

# Portfolio MtM = swaption value + IRS value
ptf_mtm = swaption_notional * swaption_price + irs_notional * irs_mtm
print(f"Swaption price (per unit notional): {swaption_price:.6f}")
print(f"IRS MtM (per unit notional):        {irs_mtm:.6f}")
print(f"Portfolio MtM:                       \u20ac{ptf_mtm:,.2f}")

Swaption price (per unit notional): 0.115955
IRS MtM (per unit notional):        0.000000
Portfolio MtM:                       €75,370,456.41


## Q2: Evaluate the portfolio DV01-parallel


In [31]:
# =============================================================================
# Q2: Portfolio DV01-parallel (numerical)
# =============================================================================

shock = 1e-4

# bootstrap returns a tuple: discount_factors and zero_rates

discount_factors_up, _ = bootstrap(settlement_date, df_depos, df_futures, 
                                   df_swaps, shock)

fwd_swap_rate_up = swap_par_rate(
    swaption_underlying_fixed_leg_schedule[1:],
    discount_factors_up,
    swaption_underlying_fixed_leg_schedule[0],
)

# Swaption price under the shocked curve
swaption_price_up = swaption_price_calculator(
    fwd_swap_rate_up, # 'up' was added 
    strike,
    settlement_date, # it was today, changed to settlemnt date
    swaption_expiry,
    underlying_expiry,
    sigma_black,
    swaption_fixed_leg_freq,
    discount_factors_up,
    swaption_type,
)

irs_mtm_up = swap_mtm(  # 'up' was added
    irs_rate, irs_fixed_leg_payment_dates, discount_factors_up, swap_type=irs_type
) 

# Shocked portfolio MtM
ptf_mtm_up = swaption_notional * swaption_price_up + irs_notional * irs_mtm_up

# DV01-parallel
ptf_numeric_dv01 = ptf_mtm_up - ptf_mtm
print(f"Portfolio DV01-parallel: \u20ac{ptf_numeric_dv01:,.2f}")

Portfolio DV01-parallel: €-367,999.97


## Q3: Analytical portfolio DV01 approximation


In [32]:
# =============================================================================
# Q3: Analytical portfolio DV01
# =============================================================================

irs_duration = irs_proxy_duration(
    settlement_date, irs_rate, irs_fixed_leg_payment_dates, discount_factors
)

ptf_proxy_dv01 = (
    swaption_notional * swaption_delta + irs_notional * irs_duration
) * shock

print(f"Portfolio proxy DV01:    \u20ac{ptf_proxy_dv01:,.2f}")
print(f"Portfolio numeric DV01:  \u20ac{ptf_numeric_dv01:,.2f}")
print(f"Difference:             \u20ac{ptf_proxy_dv01 - ptf_numeric_dv01:,.2f}")

Portfolio proxy DV01:    €-294,501.17
Portfolio numeric DV01:  €-367,999.97
Difference:             €73,498.80


## Q4: Delta-hedge the portfolio with a 10y IRS


In [ ]:
# =============================================================================
# Q4: Delta hedging with a 10y IRS
# =============================================================================

min_lot = 1_000_000  # IRS traded in multiples of €1M

# numerical hedge 
# N* = -DV01(portfolio) / DV01(10y IRS per €1M)
irs_dv01_per_1M = min_lot * irs_mtm_up 

delta_hedge_swap_notional = round(-ptf_numeric_dv01 / irs_dv01_per_1M) * min_lot

# Verify: hedged portfolio DV01 should be ~0
delta_hedge_dv01 = (
    swaption_notional * swaption_price_up + (delta_hedge_swap_notional + irs_notional) * irs_mtm_up
) - ptf_mtm # initial swap not given 
print(
    f"Numerical hedge: €{delta_hedge_swap_notional:,.0f} total IRS notional, residual DV01: €{delta_hedge_dv01:,.0f}"
)

# --- Analytical hedge (using the proxy DV01)
# N* = -DV01(ptf) / DV01(10y IRS per €1M)
# DV01 from point 3
# DV01 per unit notional = irs_duration * 1bp
delta_hedge_swap_notional_proxy = round(
    -ptf_proxy_dv01 / (irs_duration * 1e-4) / min_lot
) * min_lot

print(
    f"Analytical hedge: €{delta_hedge_swap_notional_proxy:,.0f} total IRS notional"
)
print(
    "\nThe analytical approximation significantly underestimates the required hedge notional."
)

Numerical hedge: €-459,000,000 total IRS notional, residual DV01: €31
Analytical hedge: €-356,000,000 total IRS notional

The analytical approximation significantly underestimates the required hedge notional.


## Q5: Portfolio coarse-grained bucket DV01 (10y and 15y buckets)


In [35]:
# =============================================================================
# Q5: Coarse-Grained Bucket DV01 construction
# =============================================================================

# Year fractions from settlement to each instrument's effective maturity
# (for futures, this is the quotation date; bootstrap handles the Expiry internally)
depo_years   = np.array([year_frac_act_x(settlement_date, d, 365) for d in df_depos.index])
future_years = np.array([year_frac_act_x(settlement_date, d, 365) for d in df_futures.index])
swap_years   = np.array([year_frac_act_x(settlement_date, d, 365) for d in df_swaps.index])
years = np.concatenate([depo_years, future_years, swap_years])

# Pillar dates in the order bootstrap expects them
all_dates = df_depos.index.append(df_futures.index).append(df_swaps.index)

# Bucket weights: linear blend between 10y and 15y pillars
#   10y bucket: full weight below 10y, fades to 0 by 15y
#   15y bucket: zero below 10y, builds to full weight by 15y
weights_10y = np.where(years <= 10, 1.0, np.where(years >= 15, 0.0, (15 - years) / 5))
weights_15y = 1.0 - weights_10y

# Shocked spread series (1bp), deduplicated on index
shock_10y = pd.Series(weights_10y * 1e-4, index=all_dates)
shock_15y = pd.Series(weights_15y * 1e-4, index=all_dates)
shock_10y = shock_10y[~shock_10y.index.duplicated(keep='first')]
shock_15y = shock_15y[~shock_15y.index.duplicated(keep='first')]

# Re-bootstrap the curve under each shocked scenario
discount_factors_10y_up, _ = bootstrap(settlement_date, df_depos, df_futures, df_swaps, shock_10y)
discount_factors_15y_up, _ = bootstrap(settlement_date, df_depos, df_futures, df_swaps, shock_15y)

In [42]:
# =============================================================================
# Q5: Portfolio Coarse-Grained 10y Bucket DV01
# =============================================================================

# new fwd swap rate 
fwd_swap_rate_10y_up = swap_par_rate(
    swaption_underlying_fixed_leg_schedule[1:],
    discount_factors_10y_up,
    swaption_underlying_fixed_leg_schedule[0],
)

swaption_price_10y_up = swaption_price_calculator(
    fwd_swap_rate_10y_up, # 'up' added
    strike,
    settlement_date, # it was today
    swaption_expiry,
    underlying_expiry,
    sigma_black,
    swaption_fixed_leg_freq,
    discount_factors_10y_up,
    swaption_type,
)

# IRS MtM under 10y bucket shock
irs_mtm_10y_up = swap_mtm(
    irs_rate, irs_fixed_leg_payment_dates, discount_factors_10y_up, swap_type=irs_type
)

ptf_mtm_10y_up = (
    swaption_notional * swaption_price_10y_up + irs_notional * irs_mtm_10y_up
)

ptf_numeric_10y_dv01 = ptf_mtm_10y_up - ptf_mtm
print(f"Portfolio Coarse-Grained 10y Bucket DV01: \u20ac{ptf_numeric_10y_dv01:,.2f}")

Portfolio Coarse-Grained 10y Bucket DV01: €-923,432.52


In [47]:
# =============================================================================
# Q5: Portfolio Coarse-Grained 15y Bucket DV01
# =============================================================================


fwd_swap_rate_15y_up = swap_par_rate(
    swaption_underlying_fixed_leg_schedule[1:],
    discount_factors_15y_up,
    swaption_underlying_fixed_leg_schedule[0],
)

# Swaption price under 15y bucket shock
swaption_price_15y_up = swaption_price_calculator(
    fwd_swap_rate_15y_up, 
    strike,
    settlement_date, #it was today
    swaption_expiry,
    underlying_expiry,
    sigma_black,
    swaption_fixed_leg_freq,
    discount_factors_15y_up,
    swaption_type,
)

# IRS MtM under 15y bucket shock
irs_mtm_15y_up = swap_mtm(
    irs_rate, irs_fixed_leg_payment_dates, discount_factors_15y_up,swap_type = irs_type
)

ptf_mtm_15y_up = (
    swaption_notional * swaption_price_15y_up
    + irs_notional * irs_mtm_15y_up
)

ptf_numeric_15y_dv01 = ptf_mtm_15y_up - ptf_mtm
print(f"Portfolio Coarse-Grained 15y Bucket DV01: \u20ac{ptf_numeric_15y_dv01:,.2f}")

Portfolio Coarse-Grained 15y Bucket DV01: €555,940.03


## Q6: Delta-hedge with two liquid IRS (10y and 15y)


In [58]:
# =============================================================================
# Q6: Delta hedging with two IRS (10y and 15y)
# =============================================================================

# Hedge instruments: 10y and 15y receiver IRS, matching the macro-buckets.
# Sign convention: N > 0 = receiver, N < 0 = payer (same as the original 550M IRS).

# --- DV01 sensitivities of the two hedge IRS ---
# IRS 1: 10y receiver (reuses existing schedule and rate)
irs1_dv01_10y = swap_mtm(irs_rate, irs_fixed_leg_payment_dates, discount_factors_10y_up, swap_type=SwapType.RECEIVER)
irs1_dv01_15y = swap_mtm(irs_rate, irs_fixed_leg_payment_dates, discount_factors_15y_up, swap_type=SwapType.RECEIVER)

# IRS 2: 15y receiver (build schedule and par rate from scratch)
irs2_expiry                  = business_date_offset(settlement_date, year_offset=15)
irs2_fixed_leg_payment_dates = date_series(settlement_date, irs2_expiry, irs_fixed_leg_freq)[1:]
irs2_rate                    = swap_par_rate(irs2_fixed_leg_payment_dates, discount_factors)
irs2_dv01_10y = swap_mtm(irs2_rate, irs2_fixed_leg_payment_dates, discount_factors_10y_up, swap_type=SwapType.RECEIVER)
irs2_dv01_15y = swap_mtm(irs2_rate, irs2_fixed_leg_payment_dates, discount_factors_15y_up, swap_type=SwapType.RECEIVER)

# --- Solve 2x2 system: find notionals N1, N2 that zero out both bucket DV01s ---
# A @ [N1, N2] = b  =>  ptf_dv01 + N1*irs1_dv01 + N2*irs2_dv01 = 0 for each bucket
A = np.array([
    [irs1_dv01_10y, irs2_dv01_10y],
    [irs1_dv01_15y, irs2_dv01_15y],
])
b = -np.array([ptf_numeric_10y_dv01, ptf_numeric_15y_dv01])
N1, N2 = np.linalg.solve(A, b)

# Round to nearest tradeable lot
N1 = round(N1 / min_lot) * min_lot
N2 = round(N2 / min_lot) * min_lot
print(f"Hedge IRS 1 (10y): €{N1:,.0f} ({'receiver' if N1 > 0 else 'payer'})")
print(f"Hedge IRS 2 (15y): €{N2:,.0f} ({'receiver' if N2 > 0 else 'payer'})")

# --- Verify: residual bucket DV01 after hedge ---
residual_10y = ptf_numeric_10y_dv01 + N1 * irs1_dv01_10y + N2 * irs2_dv01_10y
residual_15y = ptf_numeric_15y_dv01 + N1 * irs1_dv01_15y + N2 * irs2_dv01_15y
print(f"\nResidual 10y bucket DV01: €{residual_10y:,.0f}")
print(f"Residual 15y bucket DV01: €{residual_15y:,.0f}")

Hedge IRS 1 (10y): €-1,154,000,000 (payer)
Hedge IRS 2 (15y): €517,000,000 (receiver)

Residual 10y bucket DV01: €336
Residual 15y bucket DV01: €-249


## Q7: Curve flattening scenario


In [60]:
# =============================================================================
# Q7: Curve flattening scenario
# =============================================================================

# Flattening shock: long 10y bucket, short 15y bucket
shock_flatten = shock_10y - shock_15y
discount_factors_flatten, _ = bootstrap(settlement_date, df_depos, df_futures, df_swaps, shock_flatten)

# Reprice all instruments under the flattened curve
fwd_swap_rate_flatten = swap_par_rate(
    swaption_underlying_fixed_leg_schedule[1:],
    discount_factors_flatten,
    swaption_underlying_fixed_leg_schedule[0],
)
swaption_price_flatten = swaption_price_calculator(
    fwd_swap_rate_flatten, strike, settlement_date, swaption_expiry, underlying_expiry,
    sigma_black, swaption_fixed_leg_freq, discount_factors_flatten, swaption_type,
)
irs_mtm_flatten  = swap_mtm(irs_rate,  irs_fixed_leg_payment_dates,  discount_factors_flatten, swap_type=SwapType.RECEIVER)
irs2_mtm_flatten = swap_mtm(irs2_rate, irs2_fixed_leg_payment_dates, discount_factors_flatten, swap_type=SwapType.RECEIVER)

# P&L under flattening for each hedging strategy
# (a) swaption + single 10y receiver (550M original + delta hedge from Q4)
pnl_a = (
    swaption_notional * (swaption_price_flatten - swaption_price)
    + (irs_notional + delta_hedge_swap_notional) * irs_mtm_flatten
)

# (b) swaption + two-IRS bucket hedge (550M + N1 in 10y, N2 in 15y, from Q6)
pnl_b = (
    swaption_notional * (swaption_price_flatten - swaption_price)
    + (irs_notional + N1) * irs_mtm_flatten
    + N2 * irs2_mtm_flatten
)

print(f"Strategy (a) — single 10y IRS hedge: €{pnl_a:,.2f}")
print(f"Strategy (b) — two-IRS bucket hedge: €{pnl_b:,.2f}")
print(
    "\nStrategy (b) outperforms: hedging each bucket DV01 independently "
    "provides protection against non-parallel moves such as this flattening. "
    "Strategy (a) neutralises total DV01 only, leaving residual exposure to curve reshaping."
)

Strategy (a) — single 10y IRS hedge: €-1,111,974.88
Strategy (b) — two-IRS bucket hedge: €1,202.53

Strategy (b) outperforms: hedging each bucket DV01 independently provides protection against non-parallel moves such as this flattening. Strategy (a) neutralises total DV01 only, leaving residual exposure to curve reshaping.
